In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvZHJpZXNqL2NvZGUvZG9jdW1lbnRhdGlvbi9kb2N1bWVudGF0aW9uL0FQSXMvb3BlbkVPL2d1aWRlcw=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.14/importlib/_bootstrap.py": 1790936611.0, "/usr/lib/python3.14/importlib/_bootstrap_external.py": 1790936611.0, "/usr/lib/python3.14/zipimport.py": 1790936611.0, "/usr/lib/python3.14/codecs.py": 1790936611.0, "/usr/lib/python3.14/encodings/aliases.py": 1790936611.0, "/usr/lib/python3.14/encodings/__init__.py": 1790936611.0, "/usr/lib/python3.14/encodings/utf_8.py": 1790936611.0, "/usr/lib/python3.14/abc.py": 1790936611.0, "/usr/lib/python3.14/stat.py": 1790936611.0, "/usr/lib/python3.14/_collections_abc.py": 1790936611.0, "/usr/lib/python3.14/genericpath.py": 1790936611.0, "/usr/lib/python3.14/posixpath.py": 1790936611.0, "/usr/lib/python3.14/os.py": 1790936611.0, "/usr/lib/python3.14/_sitebuiltins.py": 1790936611.0, "/usr/lib/python3.14/site.py": 1790936611.0, "/usr/lib/python3.14/importlib/__init__.py": 1790936611.0, "/usr/lib/python3.14/importlib/machinery.py": 1790936611.0, "/usr/lib/python3.14/importlib/_abc.py": 1790936611.0, "/usr/lib/python3.14/types.py": 17

In [2]:
import openeo
from openeo import collection_property

connection = openeo.connect("openeo-staging.dataspace.copernicus.eu").authenticate_oidc()

bbox = [1397574.0, 5935182.0, 1455819.0, 5999236.0]
# Test area: 1/10 of each side, around the centre
# bbox = [1423784.25, 5964006.3, 1429608.75, 5970412.7]
extent = dict(zip(["west", "south", "east", "north"], bbox), crs="EPSG:3857")
temporal_extent = ["2026-05-01", "2026-11-01"]


def monthly_composite(coefficient: str, **backscatter_args):
    """Monthly median backscatter composite in dB, in EPSG:3857 at 10 m.
    Extra arguments are passed on to `sar_backscatter`, see the advanced parameters below."""
    cube = connection.load_collection(
        "SENTINEL1_GRD",
        spatial_extent=extent,
        temporal_extent=temporal_extent,
        bands=["VV", "VH"],
        properties=[
            collection_property("sat:orbit_state") == "ascending",
            collection_property("sar:instrument_mode") == "IW",
        ],
    )
    cube = cube.sar_backscatter(coefficient=coefficient, **backscatter_args)
    cube = cube.aggregate_temporal_period("month", reducer="median")
    cube = cube.apply(lambda x: 10 * x.log(base=10))
    return cube.resample_spatial(projection=3857, resolution=10)


Authenticated using refresh token.


In [3]:
#| echo: false
import json
from pathlib import Path

from IPython.display import HTML, Markdown
from openeo.internal.jupyter import render_component

def export_to_workspace(cube, collection: str):
    """Writes one COG per month and band, and merges their STAC items into the given collection."""
    result = cube.save_result(
        format="GTiff",
        options=dict(filename_prefix="median_composite", separate_asset_per_band=True),
    )
    return result.process(
        "export_workspace",
        arguments={"data": result, "workspace": "openeo-examples-workspace", "merge": collection},
    )

def run_job(name: str, cube, collection: str) -> dict:
    """Runs the export as a batch job once. The job id and metadata are cached in sar_processing_job_<name>.json,
    so later renders can show the job without re-running or re-fetching it."""
    job_file = Path(f"sar_processing_job_{name}.json")
    cache = json.loads(job_file.read_text()) if job_file.exists() else {}

    if "id" not in cache:
        job = export_to_workspace(cube, collection).create_job(
            title=f"SAR {name} monthly composites (docs)", job_options={"stac-version": "1.1"}, log_level="debug"
        )
        job.start_and_wait()
        cache = {"id": job.job_id, "collection": collection}

    if "metadata" not in cache:
        capabilities = connection.capabilities()
        cache["metadata"] = connection.job(cache["id"]).describe()
        cache["parameters"] = {
            "currency": capabilities.currency(),
            "federation": capabilities.ext_federation_backend_details(),
        }
        job_file.write_text(json.dumps(cache, indent=2))
    return cache


def show_job(cache: dict) -> HTML:
    return HTML(render_component("job", data=cache["metadata"], parameters=cache["parameters"]))

In [4]:
gamma0 = monthly_composite("gamma0-terrain")

In [5]:
#| echo: false
gamma0_job = run_job("gamma0", gamma0, "sar_processing_gamma0.json")
show_job(gamma0_job)

In [6]:
sigma0 = monthly_composite("sigma0-ellipsoid", options={"implementation_version": "3"})

In [7]:
#| echo: false
sigma0_job = run_job("sigma0", sigma0, "sar_processing_sigma0.json")
show_job(sigma0_job)

In [8]:
sigma0_v2 = monthly_composite("sigma0-ellipsoid", options={"implementation_version": "2"})

In [9]:
#| echo: false
sigma0_v2_job = run_job("sigma0_v2", sigma0_v2, "sar_processing_sigma0_v2.json")
show_job(sigma0_v2_job)

In [10]:
#| echo: false
import math

# Ground area of the bbox: EPSG:3857 stretches distances by 1/cos(latitude), so correct for the centre latitude.
R = 6378137.0
lat = math.atan(math.sinh((bbox[1] + bbox[3]) / 2 / R))
area_km2 = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]) * math.cos(lat) ** 2 / 1e6
start, end = (tuple(map(int, d.split("-")[:2])) for d in temporal_extent)
n_dates = (end[0] - start[0]) * 12 + end[1] - start[1]  # one composite per month
observations = area_km2 / (100 * 100) * n_dates  # 100 x 100 km observations


def usage_value(cache, key):
    entry = cache["metadata"].get("usage", {}).get(key)
    return entry["value"] if entry else None


def usage(cache, key):
    entry = cache["metadata"].get("usage", {}).get(key)
    return f"{entry['value']:,.0f} {entry['unit']}" if entry else "–"


def ratio(numerator, denominator, digits=3):
    return f"{numerator / denominator:,.{digits}f}" if numerator is not None and denominator else "–"


rows = [
    ("Credits", lambda c: f"{c['metadata'].get('costs', '–')}"),
    ("Duration", lambda c: usage(c, "duration")),
    ("CPU", lambda c: usage(c, "cpu")),
    ("Input", lambda c: usage(c, "input_pixel")),
    ("**Credits per input megapixel**", lambda c: ratio(c["metadata"].get("costs"), usage_value(c, "input_pixel"), 4)),
    ("Output observations (area × dates, in 100 × 100 km units)", lambda c: f"{observations:,.2f}"),
    ("**Credits per 100 × 100 km output observation**", lambda c: ratio(c["metadata"].get("costs"), observations, 2)),
]
jobs = [gamma0_job, sigma0_v2_job, sigma0_job]
table = ["| | γ⁰ terrain-flattened | σ⁰ ellipsoid (version 2) | σ⁰ ellipsoid (version 3) |", "|---|---|---|---|"]
table += [f"| {name} | " + " | ".join(fmt(j) for j in jobs) + " |" for name, fmt in rows]


def change(key, digits=0):
    old, new = (j["metadata"].get("costs") if key == "costs" else usage_value(j, key) for j in (sigma0_v2_job, sigma0_job))
    return f"{(new - old) / old:+.{digits}%}" if old and new is not None else "–"


table.append(
    f"| **Change of version 3 vs version 2** | | | credits {change('costs')}, duration {change('duration')}, "
    f"CPU {change('cpu')}, input {change('input_pixel')} |"
)
table.append("")
table.append(
    f"*The job produced {n_dates} dates over {area_km2:,.0f} km², i.e. {n_dates} × {area_km2 / 10000:.3f} = {observations:,.2f} "
    "observations of 100 × 100 km (10 000 km²), one observation being one date. Credits per observation = total credits / "
    "that number. The result is close to the total cost only because the number of dates and the fraction of 10 000 km² "
    "happen to nearly cancel out.*"
)
Markdown("\n".join(table))

| | γ⁰ terrain-flattened | σ⁰ ellipsoid (version 2) | σ⁰ ellipsoid (version 3) |
|---|---|---|---|
| Credits | 29 | 28 | 12 |
| Duration | 806 seconds | 804 seconds | 352 seconds |
| CPU | 7,644 cpu-seconds | 7,410 cpu-seconds | 2,376 cpu-seconds |
| Input | 2,046 mega-pixel | 2,340 mega-pixel | 2,046 mega-pixel |
| **Credits per input megapixel** | 0.0142 | 0.0120 | 0.0059 |
| Output observations (area × dates, in 100 × 100 km units) | 1.04 | 1.04 | 1.04 |
| **Credits per 100 × 100 km output observation** | 28.01 | 27.05 | 11.59 |
| **Change of version 3 vs version 2** | | | credits -57%, duration -56%, CPU -68%, input -13% |

*The job produced 6 dates over 1,725 km², i.e. 6 × 0.173 = 1.04 observations of 100 × 100 km (10 000 km²), one observation being one date. Credits per observation = total credits / that number. The result is close to the total cost only because the number of dates and the fraction of 10 000 km² happen to nearly cancel out.*

In [11]:
#| eval: false
gamma0_custom = monthly_composite(
    "gamma0-terrain",
    elevation_model="COPERNICUS_30",
    mask=True,
    options={
        "rtc_algorithm": "auto",
        "rtc_auto_min_local_incidence": 15,
        "rtc_auto_max_local_incidence": 75,
        "rtc_auto_dilation_pixels": 3,
        "cast_shadow": True,
    },
)